|                |   |
:----------------|---|
| **Nombre**     Elisa Aguirre|   |
| **Fecha**      27/04/25|   |
| **Expediente** 738894 |   |

# A08 - Bootstrapping + Aggregating

In [2]:
import pandas as pd 
import numpy as np

In [3]:
df=pd.read_excel(r"C:\Users\elisa\Downloads\Motor Trend Car Road Tests.xlsx")
df.head()

,model,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


## Regresión lineal

En esta parte se cargó el dataset **Motor Trend Car Road Tests** y se seleccionaron las variables que se usarán en el modelo. La variable respuesta es `mpg`, que representa el rendimiento del automóvil, mientras que las variables explicativas son `hp` y `qsec`.

El modelo que se ajusta es:

$$
mpg = \beta_0 + \beta_1 hp + \beta_2 qsec
$$

Después se utilizó `LinearRegression()` para ajustar la regresión lineal y obtener los coeficientes estimados del modelo: el intercepto $\beta_0$, el coeficiente de `hp` y el coeficiente de `qsec`.

In [4]:
X = df[['hp', 'qsec']]
y = df['mpg']

In [5]:
from sklearn.linear_model import LinearRegression
modelo = LinearRegression()

In [6]:
modelo.fit(X, y)

LinearRegression()

In [7]:
print("Intercepto (beta0):", modelo.intercept_)
print("Beta hp:", modelo.coef_[0])
print("Beta qsec:", modelo.coef_[1])

Intercepto (beta0): 48.32370516913445
Beta hp: -0.08459304367409272
Beta qsec: -0.8865796246342723


## Intervalos de confianza de los betas

Para calcular los intervalos de confianza al 95% de los coeficientes se utilizó `statsmodels`, ya que esta librería permite obtener información estadística del modelo, como errores estándar, valores p e intervalos de confianza.

Primero se agregó una constante a la matriz de variables explicativas para incluir el intercepto del modelo. Después se ajustó el modelo OLS y se obtuvieron los intervalos de confianza con `conf_int(alpha = 0.05)`.

El nivel de significancia usado fue:

$$
\alpha = 0.05
$$

por lo que el nivel de confianza es del 95%.

In [8]:
import statsmodels.api as sm
X = sm.add_constant(X)

# Ajustar modelo
modelo_sm = sm.OLS(y, X).fit()

# Ver resumen completo
print(modelo_sm.summary())

                            OLS Regression Results                            
Dep. Variable:                    mpg   R-squared:                       0.637
Model:                            OLS   Adj. R-squared:                  0.612
Method:                 Least Squares   F-statistic:                     25.43
Date:                Thu, 30 Apr 2026   Prob (F-statistic):           4.18e-07
Time:                        17:23:32   Log-Likelihood:                -86.170
No. Observations:                  32   AIC:                             178.3
Df Residuals:                      29   BIC:                             182.7
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         48.3237     11.103      4.352      0.0

In [9]:
intervalos = modelo_sm.conf_int(alpha=0.05)
intervalos.columns = ['LI 95%', 'LS 95%']

intervalos

,LI 95%,LS 95%
const,25.614894,71.032516
hp,-0.113089,-0.056097
qsec,-1.979929,0.206770


## Bootstrap

En esta parte se aplicó el método bootstrap para analizar la estabilidad de los coeficientes del modelo. Primero se definió:

$$
m = n
$$

donde $m$ es el número de observaciones del dataset original.

Después se crearon 1000 muestras bootstrap. Cada muestra tiene el mismo tamaño que el dataset original, pero se construye seleccionando observaciones con reemplazo. Esto significa que una misma observación puede aparecer más de una vez dentro de una muestra.

Para hacerlo sin usar una función directa de bootstrap, se generaron índices aleatorios con `np.random.choice()` y después se seleccionaron las filas correspondientes con `iloc`.

In [10]:
import numpy as np

m = len(df)

muestras_bootstrap = []

for i in range(1000):
    indices = np.random.choice(m, size=m, replace=True)
    muestra = df.iloc[indices]
    muestras_bootstrap.append(muestra)

In [11]:
muestras_bootstrap[0].head()

,model,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
1,Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
10,Merc 280C,17.8,6,167.6,123,3.92,3.440,18.90,1,0,4,4
6,Duster 360,14.3,8,360.0,245,3.21,3.570,15.84,0,0,3,4
9,Merc 280,19.2,6,167.6,123,3.92,3.440,18.30,1,0,4,4
0,Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4


## Regresión lineal en cada muestra bootstrap

Después de crear las 1000 muestras bootstrap, se ajustó una regresión lineal en cada una de ellas usando nuevamente `mpg` como variable respuesta y `hp` y `qsec` como variables explicativas.

En cada iteración se guardaron los coeficientes estimados:

$$
\beta_0,\ \beta_{hp},\ \beta_{qsec}
$$

Al final se construyó un dataframe llamado `betas_bootstrap`, donde cada fila representa los coeficientes obtenidos en una muestra bootstrap.

In [12]:
betas_bootstrap = []

for muestra in muestras_bootstrap:
    
    X_boot = muestra[['hp', 'qsec']]
    y_boot = muestra['mpg']
    
    modelo = LinearRegression()
    modelo.fit(X_boot, y_boot)
    
    betas_bootstrap.append([
        modelo.intercept_,
        modelo.coef_[0],
        modelo.coef_[1]
    ])

betas_bootstrap = pd.DataFrame(
    betas_bootstrap,
    columns=['beta0', 'beta_hp', 'beta_qsec']
)

betas_bootstrap.head()

,beta0,beta_hp,beta_qsec
0,44.280403,-0.083759,-0.711036
1,55.176771,-0.075522,-1.377069
2,58.895936,-0.097966,-1.388090
3,53.868568,-0.084629,-1.166956
4,55.859406,-0.081328,-1.314523


## Media, desviación estándar e intervalos bootstrap

Con los coeficientes obtenidos de las 1000 muestras bootstrap se calculó la media y la desviación estándar de cada beta.

La media bootstrap permite observar el valor promedio estimado de cada coeficiente, mientras que la desviación estándar mide la variabilidad de los coeficientes entre las diferentes muestras.

Los intervalos de confianza bootstrap al 95% se calcularon con:

$$
IC = \bar{\beta}_{boot} \pm 1.96(SE_{boot})
$$

Se utiliza 1.96 porque corresponde al valor crítico de la distribución normal estándar para un nivel de confianza del 95%.

In [13]:
desv_betas = betas_bootstrap.std()

desv_betas


beta0        10.676157
beta_hp       0.016716
beta_qsec     0.512770
dtype: float64

In [14]:
media_betas = betas_bootstrap.mean()

media_betas

beta0        49.933312
beta_hp      -0.087977
beta_qsec    -0.954734
dtype: float64

In [15]:
tabla_bootstrap = pd.DataFrame({
    'Media beta bootstrap': media_betas,
    'Desviación estándar': desv_betas,
    'LI 95%': media_betas - 1.96 * desv_betas,
    'LS 95%': media_betas + 1.96 * desv_betas
})

tabla_bootstrap

,Media beta bootstrap,Desviación estándar,LI 95%,LS 95%
beta0,49.933312,10.676157,29.008043,70.858580
beta_hp,-0.087977,0.016716,-0.120741,-0.055213
beta_qsec,-0.954734,0.512770,-1.959764,0.050295


## Comparación de resultados

Finalmente, se compararon los betas originales con las medias bootstrap y sus intervalos de confianza.

El coeficiente de `hp` se mantiene relativamente cercano entre el modelo original y el bootstrap, además de que su intervalo de confianza no incluye el cero. Esto sugiere que `hp` es una variable significativa para explicar `mpg`.

Por otro lado, el coeficiente de `qsec` presenta mayor variabilidad y su intervalo de confianza incluye el cero, por lo que no hay suficiente evidencia para considerarlo significativo dentro del modelo.

En general, el bootstrap permite evaluar la estabilidad de los coeficientes y observar qué tan sensibles son las estimaciones al tomar diferentes muestras con reemplazo del dataset original.

In [16]:
betas_originales = pd.Series({
    'beta0': modelo.intercept_,
    'beta_hp': modelo.coef_[0],
    'beta_qsec': modelo.coef_[1]
})

tabla_comparacion = pd.DataFrame({
    'Beta original': betas_originales,
    'Media bootstrap': media_betas,
    'Desv. estándar bootstrap': desv_betas,
    'LI 95% bootstrap': media_betas - 1.96 * desv_betas,
    'LS 95% bootstrap': media_betas + 1.96 * desv_betas
})

tabla_comparacion

,Beta original,Media bootstrap,Desv. estándar bootstrap,LI 95% bootstrap,LS 95% bootstrap
beta0,66.624588,49.933312,10.676157,29.008043,70.858580
beta_hp,-0.093752,-0.087977,0.016716,-0.120741,-0.055213
beta_qsec,-1.870046,-0.954734,0.512770,-1.959764,0.050295


In [17]:
intervalos

,LI 95%,LS 95%
const,25.614894,71.032516
hp,-0.113089,-0.056097
qsec,-1.979929,0.206770


# Train-Test

In [18]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['mpg', 'model'])
y = df['mpg']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, random_state=42)

In [35]:
columnas = X.columns
resultados = []
for i in range(1000):
    vars_random = np.random.choice(columnas, size=3, replace=False)
    Xi_train = X_train[list(vars_random)]
    modelo = LinearRegression()
    modelo.fit(Xi_train, y_train)
    resultados.append({
        'variables': vars_random,
        'modelo': modelo
    })


In [36]:
predicciones = []

for res in resultados:
    
    vars_modelo = res['variables']
    modelo = res['modelo']
    
    Xi_test = X_test[list(vars_modelo)]
    
    y_pred = modelo.predict(Xi_test)
    
    predicciones.append(y_pred)

predicciones = np.array(predicciones)

predicciones.shape

(1000, 16)

In [37]:
y_pred_promedio = predicciones.mean(axis=0)
y_pred_promedio

array([20.27462297, 10.21201531, 14.45835609, 27.15765101, 23.4372386 ,
       20.18539267, 13.43788373, 27.50661284, 15.29933254, 21.84866445,
       15.46925917, 10.4280502 , 19.78360983, 15.25799271, 14.76867198,
       13.47784965])

In [39]:
from sklearn.metrics import r2_score
r2_promedio = r2_score(y_test, y_pred_promedio)
r2_promedio

0.7847705116547194

In [31]:
resultados[37]["modelo"].predict(X_test[resultados[37]["variables"]])

array([21.9253562 , 15.09305449, 13.8147598 , 26.28482538, 31.97904718,
       24.3424815 , 20.43124553, 25.33855529, 14.72782744, 18.17718075,
       13.76495611, 14.4290053 , 22.18802411, 15.39187663, 14.39580284,
       11.09215813])

In [25]:
resultados_df = pd.DataFrame(resultados)

resultados_df.head()

,variables,r2_test
0,"[wt, disp, cyl]",0.650620
1,"[wt, carb, cyl]",0.667879
2,"[hp, wt, carb]",0.568515
3,"[drat, am, disp]",0.579067
4,"[gear, cyl, vs]",0.624881


In [24]:
mejor_modelo = resultados_df.sort_values(by='r2_test', ascending=False)

mejor_modelo.head()

,variables,r2_test
529,"[carb, disp, am]",0.777496
632,"[carb, am, disp]",0.777496
963,"[carb, am, disp]",0.777496
567,"[am, disp, carb]",0.777496
928,"[am, disp, carb]",0.777496
